In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

Libraries loaded!


In [2]:
# Load the merged dataset
df = pd.read_csv("../data/processed/diabetic_with_sdoh.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Readmission rate: {df['readmitted_30'].mean()*100:.1f}%")

Rows: 101,766
Columns: 48
Readmission rate: 11.2%


In [3]:
## --- Medication feature engineering ---
diabetes_meds = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
                 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
                 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
                 'miglitol', 'troglitazone', 'tolazamide', 'insulin',
                 'glyburide-metformin', 'glipizide-metformin',
                 'glimepiride-pioglitazone', 'metformin-rosiglitazone',
                 'metformin-pioglitazone']

# Count of active medications
df['total_diabetes_meds'] = (df[diabetes_meds] > 0).sum(axis=1)

# Medication complexity score
df['med_complexity_score'] = (
    df['num_medications'] * 0.4 +
    df['total_diabetes_meds'] * 0.3 +
    df['num_procedures'] * 0.2 +
    df['number_diagnoses'] * 0.1
)

# High medication burden flag
med_threshold = df['num_medications'].quantile(0.75)
df['high_med_burden'] = (df['num_medications'] >= med_threshold).astype(int)

# Insulin flag
df['on_insulin'] = (df['insulin'] > 0).astype(int)

# Multiple drug changes flag
df['multiple_med_changes'] = (df['change'] > 0).astype(int)

print("Medication features engineered!")
print(f"  total_diabetes_meds  — mean: {df['total_diabetes_meds'].mean():.2f}")
print(f"  med_complexity_score — mean: {df['med_complexity_score'].mean():.2f}")
print(f"  high_med_burden — {df['high_med_burden'].mean()*100:.1f}%")
print(f"  on_insulin — {df['on_insulin'].mean()*100:.1f}%")
print(f"  multiple_med_changes — {df['multiple_med_changes'].mean()*100:.1f}%")

Medication features engineered!
  total_diabetes_meds  — mean: 12.86
  med_complexity_score — mean: 11.28
  high_med_burden — 27.1%
  on_insulin — 88.0%
  multiple_med_changes — 53.8%


In [4]:
## --- Clinical risk feature engineering ---

# Prior utilization score
df['prior_utilization'] = (
    df['number_inpatient'] * 0.5 +
    df['number_emergency'] * 0.3 +
    df['number_outpatient'] * 0.2
)

# High prior inpatient flag
df['high_prior_inpatient'] = (df['number_inpatient'] >= 2).astype(int)

# Emergency admission flag
df['emergency_admission'] = (df['admission_source_id'] == 7).astype(int)

# Long stay flag
stay_threshold = df['time_in_hospital'].quantile(0.75)
df['long_stay'] = (df['time_in_hospital'] >= stay_threshold).astype(int)

# High diagnosis burden
df['high_diagnosis_burden'] = (df['number_diagnoses'] >= 7).astype(int)

print("Clinical risk features engineered!")
print(f"  prior_utilization — mean: {df['prior_utilization'].mean():.2f}")
print(f"  high_prior_inpatient — {df['high_prior_inpatient'].mean()*100:.1f}%")
print(f"  emergency_admission  — {df['emergency_admission'].mean()*100:.1f}%")
print(f"  long_stay — {df['long_stay'].mean()*100:.1f}%")
print(f"  high_diagnosis_burden — {df['high_diagnosis_burden'].mean()*100:.1f}%")

Clinical risk features engineered!
  prior_utilization — mean: 0.45
  high_prior_inpatient — 14.4%
  emergency_admission  — 56.5%
  long_stay — 28.2%
  high_diagnosis_burden — 69.4%


In [5]:
##  Feature selection (remove near-zero variance)
# Check near-zero variance features
variance = df.select_dtypes(include='number').var()
low_var = variance[variance < 0.01].index.tolist()

print(f"Near-zero variance features found: {len(low_var)}")
print(low_var)

Near-zero variance features found: 13
['nateglinide', 'chlorpropamide', 'acetohexamide', 'tolbutamide', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [6]:
# Drop low variance features (excluding target)
cols_to_drop = [c for c in low_var if c != 'readmitted_30']
df.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} near-zero variance features")
print(f"Dataset shape after dropping: {df.shape}")
print(f"Remaining columns ({df.shape[1]}):")
print(df.columns.tolist())

Dropped 13 near-zero variance features
Dataset shape after dropping: (101766, 45)
Remaining columns (45):
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'metformin', 'repaglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'change', 'diabetesMed', 'readmitted_30', 'BPHIGH', 'CHECKUP', 'CHOLSCREEN', 'DEPRESSION', 'DIABETES', 'OBESITY', 'total_diabetes_meds', 'med_complexity_score', 'high_med_burden', 'on_insulin', 'multiple_med_changes', 'prior_utilization', 'high_prior_inpatient', 'emergency_admission', 'long_stay', 'high_diagnosis_burden']


In [7]:
# Save engineered dataset
df.to_csv("../data/processed/diabetic_engineered.csv", index=False)

print("Engineered dataset saved!")
print(f"Location: data/processed/diabetic_engineered.csv")
print(f"Shape: {df.shape}")
print(f"Feature groups:")
print(f"  Clinical features:   16")
print(f"  SDOH features:        6")
print(f"  Medication features:  5")
print(f"  Risk features:        5")
print(f"  Remaining med cols:   8")
print(f"  Target:               1")
print(f"  Total:               {df.shape[1]}")

Engineered dataset saved!
Location: data/processed/diabetic_engineered.csv
Shape: (101766, 45)
Feature groups:
  Clinical features:   16
  SDOH features:        6
  Medication features:  5
  Risk features:        5
  Remaining med cols:   8
  Target:               1
  Total:               45
